# SQL Server 2019 Data Virtualization - Using Polybase to query SQL Server
This notebook contains an example of how to use external tables to query data in SQL Server (using SQL Server 2008R2 as an example) without moving data. You may need to change identity, secret, connection, database, schema, and remote table names to work with your SQL Server.

This notebook also assumes you are using SQL Server 2019 Release Candidate or later and that the Polybase feature has been installed and enabled.

This notebook uses the sample WideWorldImporters sample database but can be used with any user database.

## Step 0: Create a database in SQL Server, table, and add data

Create a database in SQL Server called **JustWorldImporters**. Execute the following T-SQL to create partitions, tables, and add data. By using partitions, SQL Server can use Polybase Scale Out Groups to read partitions in parallel if necessary.

```sql
USE [JustWorldImporters]
GO
-- Create a partition function
--
CREATE PARTITION FUNCTION [PF_Supplier_Names](nvarchar(100))
AS RANGE RIGHT FOR VALUES (N'Brooks Brothers', N'Old Suppliers -1', N'Old Suppliers -250000', 
N'Old Suppliers -500000', N'Old Suppliers -750000')
GO
-- Create the partition scheme
--
CREATE PARTITION SCHEME [PS_Supplier_Names] 
AS PARTITION [PF_Supplier_Names] 
TO ([PRIMARY], [PRIMARY], [PRIMARY], [PRIMARY], [PRIMARY], [PRIMARY])
GO
-- Create the table
--
DROP TABLE [Suppliers]
GO
CREATE TABLE [Suppliers](
	[SupplierID] [int] NOT NULL,
	[SupplierName] [nvarchar](100) NOT NULL,
	[SupplierCategoryID] [int] NOT NULL,
	[PrimaryContactPersonID] [int] NOT NULL,
	[AlternateContactPersonID] [int] NOT NULL,
	[DeliveryMethodID] [int] NULL,
	[DeliveryCityID] [int] NOT NULL,
	[PostalCityID] [int] NOT NULL,
	[SupplierReference] [nvarchar](20) NULL,
	[BankAccountName] [nvarchar](50) NULL,
	[BankAccountBranch] [nvarchar](50) NULL,
	[BankAccountCode] [nvarchar](20) NULL,
	[BankAccountNumber] [nvarchar](20) NULL,
	[BankInternationalCode] [nvarchar](20) NULL,
	[PaymentDays] [int] NOT NULL,
--	[InternalComments] [nvarchar](max) NULL,
	[PhoneNumber] [nvarchar](20) NOT NULL,
	[FaxNumber] [nvarchar](20) NOT NULL,
	[WebsiteURL] [nvarchar](256) NOT NULL,
	[DeliveryAddressLine1] [nvarchar](60) NOT NULL,
	[DeliveryAddressLine2] [nvarchar](60) NULL,
	[DeliveryPostalCode] [nvarchar](10) NOT NULL,
--  [DeliveryLocation] [geography] NULL,
	[PostalAddressLine1] [nvarchar](60) NOT NULL,
	[PostalAddressLine2] [nvarchar](60) NULL,
	[PostalPostalCode] [nvarchar](10) NOT NULL,
	[LastEditedBy] [int] NOT NULL
 CONSTRAINT [PK_Purchasing_Suppliers] PRIMARY KEY CLUSTERED 
(
	[SupplierName] ASC
) ON [PS_Supplier_names]([SupplierName]),
-- CONSTRAINT [UQ_Purchasing_Suppliers_ID] UNIQUE NONCLUSTERED 
--(
--	[SupplierID] ASC
--)
)
-- Insert some data
--
TRUNCATE TABLE [Suppliers]
GO
INSERT INTO [Suppliers]
VALUES (-1, 'Brooks Brothers', 4, -1, -2, 1, 24161, 24161, 'First US Clothing', 'Bank of New York Mellon', 'New York', NULL, '123456789', NULL, 30, '2121111111', '2121112222', 'brooksbrothers.com', '1 Broadway', NULL, '10004', '1 Broadway', NULL, '10004', 1)
GO

-- Let's go insert 1M fake suppliers
--
SET NOCOUNT ON
GO
BEGIN TRAN
GO
DECLARE @x int
DECLARE @y nvarchar(100)
SET @x = -2
WHILE @x > -1000001
BEGIN
	SET @y = 'Old Supplier'+CAST(@X as nvarchar(10))
	INSERT INTO Suppliers VALUES (@x, @Y, 4, -1, -2, 1, 24161, 24161, 'Unknown', 'Unknown', 'Unknown', NULL, '123456789', NULL, 0, '2121111111', '2121112222', 'Unknown', 'Unknown', NULL, '00000', 'Unknown', NULL, '00000', 1)
	SET @x = @x - 1
END
GO
COMMIT TRAN
GO
SET NOCOUNT OFF
GO
```


## Step 1: Create a master key
Create a master key to encrypt the database credential

In [2]:
USE [WideWorldImporters]
GO
CREATE MASTER KEY ENCRYPTION BY PASSWORD = 'S0me!nfo'
GO

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.038

## Step 2: Create a database credential.
The database credential contains the IDENTITY (login) and SECRET (password) of the remote SQL Server. Change this to the login and password for your server.

In [3]:
CREATE DATABASE SCOPED CREDENTIAL SqlServerCredentials   
WITH IDENTITY = 'sqluser', Secret = 'Sql2008R2wasagoodrelease'
GO

Commands completed successfully.

Total execution time: 00:00:00.004

## Step 3: Create an EXTERNAL DATA SOURCE
The EXTERNAL DATA SOURCE indicates what type of data source, the connection "string", where PUSHDOWN predicates should be used (if possible), and the name of the database credential.

The LOCATION syntax is <datasourcetype>:<connection string>.

datasourcetype can be sqlserver, oracle, teradata, mongodb, or odbc (Windows only)
The connection string depends on the datasourcetype

For this example, put in the name of the host of the SQL Server instance.

In [4]:
CREATE EXTERNAL DATA SOURCE SQLServerInstance
WITH ( 
LOCATION = 'sqlserver://bwsql2008r2',
PUSHDOWN = ON,
CREDENTIAL = SQLServerCredentials,
CONNECTION_OPTIONS = 'UseDefaultEncryptionOptions=false' -- This is a workaround for a bug in SQL 2019 CTP 2.3 under investigation.
)
GO

Commands completed successfully.

Total execution time: 00:00:00.027

## Step 4: Create a schema for the EXTERNAL TABLE
Schemas provide convenient methods to secure and organize objects

In [5]:
CREATE SCHEMA sqlserver
GO

Commands completed successfully.

Total execution time: 00:00:00.005

## Step 5: Create an EXTERNAL TABLE
An external table provides metadata so SQL Server knows how to map columns to the remote table. The name of the table for the external table can be your choice. But the columns must be specified in the same order with the same name as they are defined in the remote table. Furthermore, local data types must be compatible with the remote table.

The WITH clause specifies a LOCATION. This LOCATION is different than the EXTERNAL DATA SOURCE. This LOCATION indicates the [database].[schema].[table] of the remote table. The DATA_SOURCE clauses is the name of the EXTERNAL DATA SOURCE you created earlier.

In [6]:
CREATE EXTERNAL TABLE sqlserver.suppliers
(
	[SupplierID] [int] NOT NULL,
	[SupplierName] [nvarchar](100) COLLATE SQL_Latin1_General_CP1_CI_AS NOT NULL,
	[SupplierCategoryID] [int] NOT NULL,
	[PrimaryContactPersonID] [int] NOT NULL,
	[AlternateContactPersonID] [int] NOT NULL,
	[DeliveryMethodID] [int] NULL,
	[DeliveryCityID] [int] NOT NULL,
	[PostalCityID] [int] NOT NULL,
	[SupplierReference] [nvarchar](20) COLLATE SQL_Latin1_General_CP1_CI_AS NULL,
	[BankAccountName] [nvarchar](50) COLLATE SQL_Latin1_General_CP1_CI_AS NULL,
	[BankAccountBranch] [nvarchar](50) COLLATE SQL_Latin1_General_CP1_CI_AS NULL,
	[BankAccountCode] [nvarchar](20) COLLATE SQL_Latin1_General_CP1_CI_AS NULL,
	[BankAccountNumber] [nvarchar](20) COLLATE SQL_Latin1_General_CP1_CI_AS NULL,
	[BankInternationalCode] [nvarchar](20) COLLATE SQL_Latin1_General_CP1_CI_AS NULL,
	[PaymentDays] [int] NOT NULL,
	[PhoneNumber] [nvarchar](20) COLLATE SQL_Latin1_General_CP1_CI_AS NOT NULL,
	[FaxNumber] [nvarchar](20) COLLATE SQL_Latin1_General_CP1_CI_AS NOT NULL,
	[WebsiteURL] [nvarchar](256) COLLATE SQL_Latin1_General_CP1_CI_AS NOT NULL,
	[DeliveryAddressLine1] [nvarchar](60) COLLATE SQL_Latin1_General_CP1_CI_AS NOT NULL,
	[DeliveryAddressLine2] [nvarchar](60) COLLATE SQL_Latin1_General_CP1_CI_AS NULL,
	[DeliveryPostalCode] [nvarchar](10) COLLATE SQL_Latin1_General_CP1_CI_AS NOT NULL,
	[PostalAddressLine1] [nvarchar](60) COLLATE SQL_Latin1_General_CP1_CI_AS NOT NULL,
	[PostalAddressLine2] [nvarchar](60) COLLATE SQL_Latin1_General_CP1_CI_AS NULL,
	[PostalPostalCode] [nvarchar](10) COLLATE SQL_Latin1_General_CP1_CI_AS NOT NULL,
	[LastEditedBy] [int] NOT NULL
)
 WITH (
 LOCATION='JustWorldImporters.dbo.Suppliers',
 DATA_SOURCE=SqlServerInstance
)
GO

Commands completed successfully.

Total execution time: 00:00:00.092

## Step 6: Create statistics
SQL Server allows you to store local statistics about specific columns from the remote table. This can help the query processing to make more efficient plan decisions.

In [7]:
CREATE STATISTICS SupplierNameStatistics ON sqlserver.suppliers ([SupplierName]) WITH FULLSCAN
GO

Commands completed successfully.

Total execution time: 00:00:02.349

## Step 7: Try to scan the remote table
Run a simple query on the EXTERNAL TABLE to scan all rows. For this example, SQL Server should use Polybase Scale Out Groups to read partitions in parallel.

In [8]:
SELECT * FROM sqlserver.suppliers
GO

(1000000 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:17.740

SupplierID,SupplierName,SupplierCategoryID,PrimaryContactPersonID,AlternateContactPersonID,DeliveryMethodID,DeliveryCityID,PostalCityID,SupplierReference,BankAccountName,BankAccountBranch,BankAccountCode,BankAccountNumber,BankInternationalCode,PaymentDays,PhoneNumber,FaxNumber,WebsiteURL,DeliveryAddressLine1,DeliveryAddressLine2,DeliveryPostalCode,PostalAddressLine1,PostalAddressLine2,PostalPostalCode,LastEditedBy
-101020,Old Supplier-101020,4,-1,-2,1,24161,24161,Unknown,Unknown,Unknown,NULL,123456789,NULL,0,2121111111,2121112222,Unknown,Unknown,NULL,00000,Unknown,NULL,00000,1
-101021,Old Supplier-101021,4,-1,-2,1,24161,24161,Unknown,Unknown,Unknown,NULL,123456789,NULL,0,2121111111,2121112222,Unknown,Unknown,NULL,00000,Unknown,NULL,00000,1
-101022,Old Supplier-101022,4,-1,-2,1,24161,24161,Unknown,Unknown,Unknown,NULL,123456789,NULL,0,2121111111,2121112222,Unknown,Unknown,NULL,00000,Unknown,NULL,00000,1
-101023,Old Supplier-101023,4,-1,-2,1,24161,24161,Unknown,Unknown,Unknown,NULL,123456789,NULL,0,2121111111,2121112222,Unknown,Unknown,NULL,00000,Unknown,NULL,00000,1
-101024,Old Supplier-101024,4,-1,-2,1,24161,24161,Unknown,Unknown,Unknown,NULL,123456789,NULL,0,2121111111,2121112222,Unknown,Unknown,NULL,00000,Unknown,NULL,00000,1
-101025,Old Supplier-101025,4,-1,-2,1,24161,24161,Unknown,Unknown,Unknown,NULL,123456789,NULL,0,2121111111,2121112222,Unknown,Unknown,NULL,00000,Unknown,NULL,00000,1
-101026,Old Supplier-101026,4,-1,-2,1,24161,24161,Unknown,Unknown,Unknown,NULL,123456789,NULL,0,2121111111,2121112222,Unknown,Unknown,NULL,00000,Unknown,NULL,00000,1
-101027,Old Supplier-101027,4,-1,-2,1,24161,24161,Unknown,Unknown,Unknown,NULL,123456789,NULL,0,2121111111,2121112222,Unknown,Unknown,NULL,00000,Unknown,NULL,00000,1
-101028,Old Supplier-101028,4,-1,-2,1,24161,24161,Unknown,Unknown,Unknown,NULL,123456789,NULL,0,2121111111,2121112222,Unknown,Unknown,NULL,00000,Unknown,NULL,00000,1
-101029,Old Supplier-101029,4,-1,-2,1,24161,24161,Unknown,Unknown,Unknown,NULL,123456789,NULL,0,2121111111,2121112222,Unknown,Unknown,NULL,00000,Unknown,NULL,00000,1


## Step 8: Query the remote table with a WHERE clause
Even though the table may be small SQL Server will "push" the WHERE clause filter to the remote table

In [9]:
SELECT * FROM sqlserver.suppliers where SupplierName = 'Brooks Brothers'
GO

(1 row affected)

Total execution time: 00:00:00.317

SupplierID,SupplierName,SupplierCategoryID,PrimaryContactPersonID,AlternateContactPersonID,DeliveryMethodID,DeliveryCityID,PostalCityID,SupplierReference,BankAccountName,BankAccountBranch,BankAccountCode,BankAccountNumber,BankInternationalCode,PaymentDays,PhoneNumber,FaxNumber,WebsiteURL,DeliveryAddressLine1,DeliveryAddressLine2,DeliveryPostalCode,PostalAddressLine1,PostalAddressLine2,PostalPostalCode,LastEditedBy
-1,Brooks Brothers,4,-1,-2,1,24161,24161,First US Clothing,Bank of New York Mellon,New York,NULL,123456789,NULL,30,2121111111,2121112222,brooksbrothers.com,1 Broadway,NULL,10004,1 Broadway,NULL,10004,1


## Step 9: Join with local SQL Server tables
Find all former clothing suppliers and their city.

In [10]:
SELECT s.SupplierName, s.SupplierReference, c.cityname
FROM sqlserver.suppliers s
JOIN [Purchasing].[SupplierCategories] sc
on s.SupplierCategoryID = sc.SupplierCategoryID
and sc.SupplierCategoryName = 'Clothing Supplier'
JOIN [Application].[Cities] c
ON s.DeliveryCityID = c.CityID
GO

(1000000 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:05.953

SupplierName,SupplierReference,cityname
Old Supplier-101863,Unknown,New York
Old Supplier-101864,Unknown,New York
Old Supplier-101865,Unknown,New York
Old Supplier-101866,Unknown,New York
Old Supplier-101867,Unknown,New York
Old Supplier-101868,Unknown,New York
Old Supplier-101869,Unknown,New York
Old Supplier-10187,Unknown,New York
Old Supplier-101870,Unknown,New York
Old Supplier-101871,Unknown,New York
